In [1]:
#| include: false
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from utils.style import COLORS, CYCLE, apply_style

apply_style()
np.set_printoptions(precision=4, suppress=True, linewidth=110)
_ = pl.Config.set_float_precision(4)

We use the Household Budget Survey (EPF) from INE (Instituto Nacional de Estadística) introduced in
[the first part of the topic](01-stats-population-samples.ipynb#parameters-and-statistics).
The file has many variables (they are described in `data/processed/epf/README.md`);
here we keep seven of them:

In [2]:
households = pl.read_csv("data/processed/epf/households.csv")

households = households.select(
    "household_id", "region", "tenure_group",
    "household_size", "floor_area_m2", 
    "monthly_income", "main_income_source",
    "exp_total", "exp_monetary", "exp_food", "exp_clothing",
    "n_employed", "ref_sex", "ref_age", "ref_education"
)

The fifteen variables are:

- `household_id`: identifier of the household in the survey.
- `region`: autonomous community of residence (the 17 regions plus the
  autonomous cities of Ceuta and Melilla).
- `tenure_group`: tenure of the main dwelling: Owned, Rented, or Other (a
  dwelling provided free or at a reduced cost, for example by a relative or an
  employer).
- `household_size`: number of members of the household.
- `floor_area_m2`: useful floor area of the main dwelling, in m². For
  confidentiality, every dwelling of 35 m² or less is recorded as 35, and every
  dwelling of 300 m² or more as 300.
- `monthly_income`: net **monthly** income of the household, in euros (after
  taxes and social contributions). It is 0 for the few households with no regular
  income.
- `main_income_source`: main source of the household's income: Employee
  earnings, Self-employment and property income, or Pensions and benefits.
- `exp_total`: total **annual** expenditure of the household, in euros. Besides
  what the household actually pays, it includes some *non-monetary* items. The
  most important is the *imputed rent*: for households that own their home (or
  live in one provided free), the INE adds the rent they would pay if they had to
  rent a similar dwelling. This makes owners and tenants comparable.
- `exp_monetary`: the part of `exp_total` that the household actually pays, in
  euros. It excludes the non-monetary items, so for owners it does not include
  the imputed rent.
- `exp_food`: annual expenditure on food and non-alcoholic beverages, in euros.
- `exp_clothing`: annual expenditure on clothing and footwear, in euros.
- `n_employed`: number of members of the household who have a job.
- `ref_sex`, `ref_age` and `ref_education`: sex, age and highest education
  level completed (four levels, from "Below lower secondary" to "Tertiary") of
  the household's *reference person*, the member who contributes most to its
  budget. Age is in completed years, and everyone aged 85 or more is recorded
  as 85.


## Variables and data

::: {.callout-note}
## Definition

- A **variable** (or attribute) is a characteristic of the individuals that can
  take different **values**, or **categories**, from one individual to another (this is why it is called variable and not constant).
- **Data** are the observed values of one or more variables on a set of
  individuals.
- We denote by $x_i$ the value of the variable $X$ for individual $i$,
$i = 1, \dots, n$.
:::

Data are organised in a **data matrix**, in which **each row is an individual and
each column a variable**. This is exactly the structure of a Polars `DataFrame`
(and the *tidy data* layout):

In [3]:
households.head()

household_id,region,tenure_group,household_size,floor_area_m2,monthly_income,main_income_source,exp_total,exp_monetary,exp_food,exp_clothing,n_employed,ref_sex,ref_age,ref_education
1,Catalonia,Owned,4,85,"4,000.0",Employee earnings,"36,243.1","29,702.8","11,124.5","1,016.4",2,Male,45,Tertiary
2,Valencian Community,Owned,2,280,"7,614.0",Pensions and benefits,"53,947.3","37,922.6","3,460.8",0.000,0,Male,75,Tertiary
3,Galicia,Rented,2,90,"2,213.0",Employee earnings,"21,041.8","21,041.8","2,234.5","2,532.4",1,Female,39,Upper secondary
4,Catalonia,Owned,4,70,"2,281.0",Self-employment and property income,"39,369.1","30,092.5","6,598.4","4,053.0",1,Male,47,Upper secondary
5,Navarre,Owned,2,105,"1,740.0",Pensions and benefits,"16,325.9","10,055.6","4,767.3",68.430,0,Male,75,Lower secondary




## Classification of variables

The type of a variable determines which tables, graphs and numerical measures
make sense. The basic classification is the following.

**Qualitative (categorical) variables**: their categories are qualities, not
quantities.

- **Nominal**: the categories have **no** natural order. For example, region, sector
  of activity, blood group or housing tenure.
- **Ordinal**: the categories have a natural order, but the distance between them
  is not defined. For example, education level, the stage of a disease or a
  credit rating.
- An important special case are **binary** (dichotomous) variables, with only two
  categories: being unemployed or not, or having a disease or not. They are
  usually coded as 1 and 0, and their mean is the proportion of ones.

**Quantitative variables**: their values are numbers that express an amount,
measured in some units.

- **Discrete**: they take isolated values, usually integers, that can be counted.
  For example, the number of household members or the number of hospital
  admissions.
- **Continuous**: they can take, at least in theory, any value in an interval.
  For example, income, price, blood pressure or the time until a component fails.

```{mermaid}
%%| echo: false
%%| label: fig-types
%%| fig-cap: "Classification of statistical variables."
flowchart TD
    V["Variable"] --> CL["Qualitative"]
    V --> CU["Quantitative"]
    CL --> N["Nominal<br/><i>region, sector</i>"]
    CL --> O["Ordinal<br/><i>education level</i>"]
    CU --> D["Discrete<br/><i>household members</i>"]
    CU --> C["Continuous<br/><i>income, price</i>"]
```

::: {.callout-warning}
## Being recorded as a number does not make a variable quantitative

Many qualitative variables are recorded with numerical codes: postcodes, codes of
economic activity or education levels coded as 1, 2, 3.
Computing the mean of these codes makes no sense. The type of a variable is
determined by **what it measures**, not by how it is stored.
:::

::: {.callout-tip}
## Discrete or continuous in practice

Every measurement has finite precision: income is recorded in euros or cents, and
age in completed years. Strictly speaking, all observed data are discrete. In
practice, we treat a variable as **continuous** when it takes many different
values and intermediate values make sense. Income is continuous for practical
purposes; the number of children is not.
:::

## Measurement scales

A finer classification, due to Stevens [-@stevens1946], looks at the
**operations** that make sense with the values of the variable:

| Scale | What it allows | Example | Suitable measures of centre |
|---|---|---|---|
| Nominal | Equality: $=$, $\neq$ | Region | Mode |
| Ordinal | Also order: $<$, $>$ | Education level | Mode, median |
| Interval | Also differences: $x_i - x_j$ | Temperature in °C, calendar year | Mode, median, mean |
| Ratio | Also ratios: $x_i / x_j$ | Income, price, number of employees | All |

: Measurement scales and admissible operations. {#tbl-scales}

The difference between the last two scales lies in the **zero**. On a ratio scale
the zero is absolute: it means absence of the quantity (zero income, no
employees), and it makes sense to say that one household earns *twice as much*
as another. On an interval scale the zero is conventional. 20 °C is not "twice as
hot" as 10 °C, and the year 2000 is not "twice" the year 1000. Most quantitative
variables in economics and health are on a ratio scale, which allows rates of
change and relative measures such as the coefficient of variation
([Topic 2](../02-univariate/02-numerical-summaries.ipynb#measures-of-dispersion)).

::: {.callout-note}
## Historical note: S. S. Stevens and the scales of measurement

The psychologist Stanley Smith Stevens proposed the four scales in a short paper
in *Science* in 1946 [@stevens1946]. It grew out of a long debate in a committee
of the British Association for the Advancement of Science on whether sensations
such as loudness could be "measured" at all. Stevens classified scales by the
transformations that leave their meaning unchanged: any one-to-one relabelling
for nominal scales, any increasing function for ordinal ones, $a + bx$ for
interval scales and $bx$ for ratio scales. The classification is still standard,
although it has been criticised as too rigid. For example, sums of Likert items
are routinely treated as interval variables.
:::

## Types of data by structure

How the data are organised over time also matters:

- **Cross-sectional data**: many individuals observed at the same time. For
  example, the expenditure of about 20,000 households in 2025, as in our survey.
- **Time series**: a *single* individual or aggregate observed at many points
  in time. For example, the monthly consumer price index of Spain, or its
  quarterly GDP: one number per period.
- **Panel (longitudinal) data**: *many* individuals, and the *same* ones,
  observed at several points in time. It is like having one time series for each
  individual, so we can follow how each of them changes. For example, the Labour
  Force Survey interviews each household for several consecutive quarters, so we
  can see who moves from unemployment into a job.

Panel data should not be confused with **repeated cross-sections**: the EPF is
carried out every year, but mostly with *different* households each time. We can
compare 2024 with 2025, but we cannot follow a given household.

| Type | Individuals | Time points | Example |
|---|---|---|---|
| Cross-section | many | one | EPF 2025 |
| Time series | one | many | Monthly CPI |
| Panel | many, the same ones | many | Labour Force Survey |
| Repeated cross-sections | many, different ones | many | EPF 2006–2025 |

These notes mostly work with cross-sectional data. Some descriptive tools for
time series are introduced in [Topic 8](../08-index-numbers-time-series.qmd).

## Types of variables in Python

Polars assigns each column a **data type** (*dtype*), which says how it is
stored. This storage type does not necessarily coincide with the statistical
type:

In [4]:
pl.DataFrame({
    "variable": households.columns,
    "Polars type": [str(t) for t in households.dtypes],
    "statistical type": [
        "Identifier (not a study variable)",          # household_id
        "Qualitative, nominal",                       # region
        "Qualitative, nominal",                       # tenure_group
        "Quantitative, discrete",                     # household_size
        "Quantitative, continuous",                   # floor_area_m2
        "Quantitative, continuous",                   # monthly_income
        "Qualitative, nominal",                       # main_income_source
        "Quantitative, continuous",                   # exp_total
        "Quantitative, continuous",                   # exp_monetary
        "Quantitative, continuous",                   # exp_food
        "Quantitative, continuous",                   # exp_clothing
        "Quantitative, discrete",                     # n_employed
        "Qualitative, nominal (binary)",              # ref_sex
        "Quantitative, continuous (in whole years)",  # ref_age
        "Qualitative, ordinal",                       # ref_education
    ],
})

variable,Polars type,statistical type
household_id,Int64,Identifier (not a study variable)
region,String,"Qualitative, nominal"
tenure_group,String,"Qualitative, nominal"
household_size,Int64,"Quantitative, discrete"
floor_area_m2,Int64,"Quantitative, continuous"
monthly_income,Float64,"Quantitative, continuous"
main_income_source,String,"Qualitative, nominal"
exp_total,Float64,"Quantitative, continuous"
exp_monetary,Float64,"Quantitative, continuous"
exp_food,Float64,"Quantitative, continuous"


Two remarks:

- `household_id` is an integer, but it only **identifies** each household.
  Polars would happily compute its mean, but that number means nothing.
- `ref_education` is stored as text (`String`), so Polars does not know that its
  categories are ordered. If we sort them, the order is alphabetical:

In [5]:
households["ref_education"].unique().sort()

ref_education
Below lower secondary
Lower secondary
Tertiary
Upper secondary


To declare an ordinal variable in Polars we use the `pl.Enum` type, which stores the
categories **in the order given**. Sorting and comparisons then respect the
logical order:

In [6]:
education_order = pl.Enum(
    ["Below lower secondary", "Lower secondary", "Upper secondary", "Tertiary"]
)
education = households["ref_education"].cast(education_order)

education.unique().sort()

ref_education
Below lower secondary
Lower secondary
Upper secondary
Tertiary


In [7]:
# Proportion of households whose reference person has at most upper secondary education
round((education <= "Upper secondary").mean(), 2)

0.59

The last expression illustrates two ideas of this topic: the comparison $\leq$
only makes sense because the variable is **ordinal**, and the mean of a **binary**
variable (true = 1, false = 0) is a **proportion**.

## Summary of the topic

- Statistics studies how to draw conclusions from data that show
  **variability**, quantifying the **uncertainty**.
- **Descriptive statistics** summarises the observed data. **Inference**
  generalises from the sample to the population, based on **probability**.
- The **population** ($N$) is the set of interest; the **sample** ($n$) is the
  part observed. To be representative, the sample must be selected **at random**,
  and increasing its size does not correct a selection bias.
- A **parameter** describes the population, and is fixed and unknown. A
  **statistic** is computed from the sample and varies from sample to sample.
- Variables are **qualitative** (nominal or ordinal) or **quantitative** (discrete
  or continuous). The **measurement scale** determines which operations and
  summaries make sense.

## Exercises

::: {#exr-classify}
## Classifying variables
Classify the following variables by type (qualitative nominal or ordinal,
quantitative discrete or continuous) and measurement scale:

a) Sector of activity of a firm (NACE code).
b) Number of workers of a firm.
c) Annual return of an investment fund (%).
d) Credit rating of a country (AAA, AA+, AA, …).
e) Year a firm was founded.
f) Labour force status (employed, unemployed, inactive).
g) Satisfaction with a service on a scale from 1 to 5.
h) Body temperature of a patient (°C).
:::

::: {.callout-tip collapse="true"}
## Solution

a) Qualitative nominal, even though it is coded with numbers.

b) Quantitative discrete, ratio scale.

c) Quantitative continuous, ratio scale. Zero means neither gain nor loss, and a
4 % return is twice the gain of a 2 % return. Negative values do not prevent it
from being a ratio scale.

d) Qualitative ordinal.

e) Quantitative discrete, interval scale. Subtracting years to obtain the age of
the firm makes sense, but dividing them does not.

f) Qualitative nominal.

g) Qualitative ordinal. Treating it as quantitative and computing its mean is
common in practice, but it assumes that the distance between 1 and 2 is the same
as between 4 and 5.

h) Quantitative continuous, interval scale: 0 °C is not an absence of
temperature.
:::

## References {.unnumbered}

::: {#refs}
:::